# PourCastAI — Step 1: Network Egress Test

Databricks Free Edition is serverless-only and restricts outbound internet
access. Before we build any ingestion pipeline, we need to know whether THIS
workspace's compute can reach the same APIs `risk_tools.py` already calls
locally (OSRM, NWS, Open-Meteo, EIA), plus the HubSpot API we're about to add.

Run this whole notebook once (Run All). Read the summary table at the bottom.
It tells us which of the two ingestion designs to build next.

In [0]:
import requests
import time

TIMEOUT = 8
HEADERS = {"User-Agent": "PourCastAI-student-project (contact: example@uni.edu)"}

# Same URLs risk_tools.py already calls (Ankeny -> a sample store, for OSRM),
# plus HubSpot's CRM API root (no auth token needed to test reachability --
# a 401 here still proves the network path is open).
tests = {
    "OSRM (routing)":        "https://router.project-osrm.org/route/v1/driving/-93.558,41.699;-93.6,41.6?overview=false",
    "NWS (weather/hazards)": "https://api.weather.gov/alerts/active?area=IA",
    "Open-Meteo (forecast)": "https://api.open-meteo.com/v1/forecast?latitude=41.6&longitude=-93.6&hourly=precipitation_probability&forecast_days=1",
    "EIA (diesel price)":    "https://api.eia.gov/v2/petroleum/pri/gnd/data/?api_key=DEMO_KEY&frequency=weekly&data[0]=value&length=1",
    "HubSpot CRM API":       "https://api.hubapi.com/crm/v3/objects/companies?limit=1",
}

In [0]:
results = []
for name, url in tests.items():
    try:
        t0 = time.time()
        r = requests.get(url, timeout=TIMEOUT, headers=HEADERS)
        elapsed = round(time.time() - t0, 2)
        # Any HTTP response (even 401/403/429) means the network path is open --
        # only a connection error / timeout means Databricks actually BLOCKED it.
        results.append((name, "REACHABLE", str(r.status_code), elapsed))
    except requests.exceptions.RequestException as e:
        results.append((name, "BLOCKED", type(e).__name__, None))

In [0]:
print(f"{'API':<26}{'Result':<12}{'HTTP/Error':<20}{'Seconds'}")
print("-" * 70)
for name, status, detail, elapsed in results:
    print(f"{name:<26}{status:<12}{detail:<20}{elapsed if elapsed is not None else ''}")

blocked = [r[0] for r in results if r[1] == "BLOCKED"]

print()
if not blocked:
    print("ALL REACHABLE. -> Build ingestion notebooks that call these APIs")
    print("directly from Databricks (Bronze layer runs IN Databricks).")
else:
    print(f"BLOCKED: {blocked}")
    print("-> Keep the API-calling code running locally (as it does today in")
    print("risk_tools.py), and have it WRITE results into Databricks instead")
    print("(via databricks-sql-connector or a Volume upload), rather than")
    print("calling these APIs from inside a Databricks notebook.")

API                       Result      HTTP/Error          Seconds
----------------------------------------------------------------------
OSRM (routing)            REACHABLE   200                 0.43
NWS (weather/hazards)     REACHABLE   200                 0.17
Open-Meteo (forecast)     REACHABLE   200                 0.49
EIA (diesel price)        BLOCKED     ReadTimeout         
HubSpot CRM API           REACHABLE   401                 0.05

BLOCKED: ['EIA (diesel price)']
-> Keep the API-calling code running locally (as it does today in
risk_tools.py), and have it WRITE results into Databricks instead
(via databricks-sql-connector or a Volume upload), rather than
calling these APIs from inside a Databricks notebook.


## What to do with this result

- **All reachable:** tell me, and Step 2 becomes "build the HubSpot Bronze
  ingestion notebook" -- everything (HubSpot pull + your 4 risk APIs) can run
  as scheduled Databricks notebooks.
- **Some/all blocked:** tell me which ones, and Step 2 becomes "keep
  `risk_tools.py` running where it runs today, add a small write-through to
  Databricks." Nothing about your existing risk scoring logic needs to
  change -- only where the Gold tables end up.